In [ ]:
# INSTALL 

!pip install -q -U chromadb sentence-transformers transformers accelerate bitsandbytes

In [ ]:
# IMPORTS

import re
import shutil
import torch
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig
from huggingface_hub import login

In [ ]:
# LOGIN HUGGING FACE


In [ ]:
# =========================
# CONFIG
# =========================

DRIVE_DB_LIBRO  = Path("/content/drive/MyDrive/NLP_PRACTICA/Persist/libro_largo_qwen")
DRIVE_DB_RESUMEN = Path("/content/drive/MyDrive/NLP_PRACTICA/Persist/chroma_db_resumenes_qwen")

LOCAL_DB_LIBRO  = Path("/content/chroma_got_rag_libro_largo_qwen")
LOCAL_DB_RESUMEN = Path("/content/chroma_got_rag_resumenes_qwen")

COLLECTION_LIBRO = "libro"
COLLECTION_RESUMEN = "resumenes_got"

EMBED_MODEL_NAME = "Qwen/Qwen3-Embedding-8B"
LLM_MODEL_NAME   = "google/gemma-4-31B-it"

TOP_K = 5
MAX_NEW_TOKENS = 256

In [ ]:

# =========================
# 1. COPIAR CHROMA DE DRIVE
# =========================

def load_chroma_from_drive() -> tuple[chromadb.Collection, chromadb.Collection]:
    if LOCAL_DB_LIBRO.exists():
        shutil.rmtree(LOCAL_DB_LIBRO)

    print("Copiando Chroma del Libro de Drive a /content...")
    shutil.copytree(DRIVE_DB_LIBRO, LOCAL_DB_LIBRO)

    if LOCAL_DB_RESUMEN.exists():
        shutil.rmtree(LOCAL_DB_RESUMEN)

    print("Copiando Chroma de Resúmenes de Drive a /content...")
    shutil.copytree(DRIVE_DB_RESUMEN, LOCAL_DB_RESUMEN)

    client_libro = chromadb.PersistentClient(path=str(LOCAL_DB_LIBRO))
    col_libro = client_libro.get_collection(COLLECTION_LIBRO)

    client_resumen = chromadb.PersistentClient(path=str(LOCAL_DB_RESUMEN))
    col_resumen = client_resumen.get_collection(COLLECTION_RESUMEN)

    print(f"  → Colección Libro cargada: {col_libro.count()} chunks")
    print(f"  → Colección Resúmenes cargada: {col_resumen.count()} chunks")

    return col_libro, col_resumen


In [ ]:
# =========================
# 2. CARGAR MODELOS
# =========================

def load_embedder() -> SentenceTransformer:
    print("Cargando embedder Qwen3-Embedding-8B...")
    return SentenceTransformer(
        EMBED_MODEL_NAME,
        model_kwargs={"torch_dtype": torch.bfloat16}
    )


def load_llm():
    print("Cargando LLM Gemma 4 31B IT en 4-bit...")

    processor = AutoProcessor.from_pretrained(LLM_MODEL_NAME)

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        quantization_config=quant_config,
        low_cpu_mem_usage=True
    )

    model.eval()

    model.generation_config = GenerationConfig(
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
        repetition_penalty=1.08,
        pad_token_id=processor.tokenizer.eos_token_id,
        eos_token_id=processor.tokenizer.eos_token_id
    )

    return processor, model



In [ ]:

# =========================
# 3. RECUPERACIÓN DOBLE
# =========================

def retrieve_from_col(col: chromadb.Collection, emb: list, top_k: int) -> list[dict]:
    try:
        results = col.query(
            query_embeddings=[emb],
            n_results=top_k,
            include=["documents", "metadatas", "distances"]
        )

        chunks = []

        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ):
            chunks.append({
                "text": doc,
                "metadata": meta,
                "score": round(1 - dist, 4)
            })

        return chunks

    except Exception as e:
        print(f"Aviso: no se pudo recuperar de una colección: {e}")
        return []


def retrieve(
    col_libro: chromadb.Collection,
    col_resumen: chromadb.Collection,
    embedder: SentenceTransformer,
    query: str,
    top_k: int = TOP_K
) -> list[dict]:

    query_prefixed = "Represent this sentence for searching relevant passages: " + query

    emb = embedder.encode(
        query_prefixed,
        normalize_embeddings=True
    ).tolist()

    chunks_libro = retrieve_from_col(col_libro, emb, top_k)
    chunks_resumen = retrieve_from_col(col_resumen, emb, top_k)

    for c in chunks_libro:
        c["source"] = "Libro"

    for c in chunks_resumen:
        c["source"] = "Web Scraper"

    all_chunks = chunks_libro + chunks_resumen
    all_chunks.sort(key=lambda x: x["score"], reverse=True)

    return all_chunks[:top_k]


In [ ]:


# =========================
# 4. CONSTRUCCIÓN DEL PROMPT
# =========================

def build_rag_prompt(query: str, chunks: list[dict]) -> tuple[str, str]:
    context_parts = []

    for i, chunk in enumerate(chunks, 1):
        meta = chunk["metadata"]
        source = chunk.get("source", "N/A")

        chapter = meta.get("chapter_title") or meta.get("chapter", "N/A")
        pov = meta.get("pov", "N/A")

        context_parts.append(
            f"[Fragmento {i} — Fuente: {source}, Capítulo: {chapter}, "
            f"POV: {pov}, Similitud: {chunk['score']}]\n"
            f"{chunk['text']}"
        )

    context = "\n\n---\n\n".join(context_parts)

    system = (
        "Eres un asistente experto en la novela Juego de Tronos. "
        "Responde en español usando ÚNICAMENTE la información de los fragmentos proporcionados. "
        "Si la respuesta no está en los fragmentos, dilo explícitamente. "
        "No inventes información. "
        "Responde de manera clara, breve y directa."
    )

    user = f"""Fragmentos recuperados del libro y de resúmenes web:

{context}

---

Pregunta: {query}

Responde basándote exclusivamente en los fragmentos anteriores."""

    return system, user


In [ ]:


# =========================
# 5. GENERACIÓN CON GEMMA 4
# =========================

def generate_answer(processor, model, query: str, chunks: list[dict]) -> str:
    system, user = build_rag_prompt(query, chunks)

    prompt = f"""<start_of_turn>user
{system}

{user}
<end_of_turn>
<start_of_turn>model
"""

    inputs = processor.tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=12000
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.08,
            eos_token_id=processor.tokenizer.eos_token_id,
            pad_token_id=processor.tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]

    answer = processor.tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return answer


In [ ]:
# =========================
# 6. EVALUACIÓN SIMPLE DE FIDELIDAD
# =========================

def simple_faithfulness_check(answer: str, chunks: list[dict], min_overlap: float = 0.35) -> dict:
    """
    Approximate faithfulness check.

    It splits the generated answer into sentences and checks whether the relevant
    words from each sentence appear in the retrieved context.
    """

    context = " ".join(chunk["text"] for chunk in chunks).lower()

    sentences = re.split(r"[.!?]\s+", answer)

    supported = []
    unsupported = []

    for sent in sentences:
        sent_clean = sent.strip()

        if not sent_clean:
            continue

        words = [
            w.lower()
            for w in re.findall(r"\w+", sent_clean)
            if len(w) > 4
        ]

        if not words:
            unsupported.append({
                "sentence": sent_clean,
                "overlap_ratio": 0.0
            })
            continue

        overlap = sum(1 for w in words if w in context)
        ratio = overlap / len(words)

        item = {
            "sentence": sent_clean,
            "overlap_ratio": round(ratio, 3)
        }

        if ratio >= min_overlap:
            supported.append(item)
        else:
            unsupported.append(item)

    total = len(supported) + len(unsupported)

    faithfulness_score = len(supported) / total if total > 0 else 0.0

    return {
        "faithfulness_score": round(faithfulness_score, 3),
        "supported_sentences": supported,
        "unsupported_sentences": unsupported
    }


In [ ]:
# =========================
# 7. IMPRESIÓN DE RESULTADOS
# =========================

def print_chunks(chunks: list[dict]):
    print("\n── Fragmentos recuperados ──────────────────────────────")

    for i, chunk in enumerate(chunks, 1):
        meta = chunk["metadata"]
        source = chunk.get("source", "N/A")

        chapter = meta.get("chapter_title") or meta.get("chapter", "N/A")
        pov = meta.get("pov", "N/A")
        chars = meta.get("characters", "N/A")

        print(
            f"  [{i}] [{source}] {chapter} | "
            f"POV: {pov} | "
            f"Score: {chunk['score']} | "
            f"Personajes: {chars}"
        )

    print("────────────────────────────────────────────────────────\n")


def print_faithfulness(faithfulness: dict):
    print("📊 Evaluación aproximada de fidelidad")
    print(f"Faithfulness score: {faithfulness['faithfulness_score']}")

    if faithfulness["unsupported_sentences"]:
        print("\n⚠️ Frases potencialmente no soportadas por los chunks:")

        for item in faithfulness["unsupported_sentences"]:
            print(f"- {item['sentence']} | overlap={item['overlap_ratio']}")
    else:
        print("\n✅ Todas las frases parecen estar razonablemente soportadas.")

    print()


In [ ]:
# =========================
# 8. ASK
# =========================

def ask(
    query: str,
    col_libro,
    col_resumen,
    embedder,
    processor,
    model,
    show_chunks: bool = True,
    show_faithfulness: bool = True
):
    print(f"\n❓ Pregunta: {query}")

    chunks = retrieve(
        col_libro=col_libro,
        col_resumen=col_resumen,
        embedder=embedder,
        query=query
    )

    if show_chunks:
        print_chunks(chunks)

    answer = generate_answer(
        processor=processor,
        model=model,
        query=query,
        chunks=chunks
    )

    print(f"💬 Respuesta:\n{answer}\n")

    faithfulness = simple_faithfulness_check(answer, chunks)

    if show_faithfulness:
        print_faithfulness(faithfulness)

    return {
        "query": query,
        "chunks": chunks,
        "answer": answer,
        "faithfulness": faithfulness
    }



In [ ]:
# =========================
# MAIN
# =========================

def main():
    col_libro, col_resumen = load_chroma_from_drive()
    embedder = load_embedder()
    processor, model = load_llm()

    print("\n✅ Sistema RAG listo. Escribe 'salir' para terminar.\n")

    while True:
        query = input("❓ Pregunta: ").strip()

        if not query:
            continue

        if query.lower() == "salir":
            break

        ask(
            query=query,
            col_libro=col_libro,
            col_resumen=col_resumen,
            embedder=embedder,
            processor=processor,
            model=model,
            show_chunks=True,
            show_faithfulness=True
        )



In [ ]:

if __name__ == "__main__":
    main()